# Worked Example: Bipolar Reference

## Goal
Apply bipolar referencing to the real recording. Artifact QC is covered in chapter 05.


In [ ]:
from pathlib import Path
from LFPAnalysis import build_basic_pipeline_config, run_pipeline

config = build_basic_pipeline_config(
    Path('../../data/sample_ieeg.fif'),
    file_format='mne',
    reference_method='bipolar',
    electrode_path=Path('../../data/sample_labels.xlsx'),
)
result = run_pipeline(config)
# After bipolar re-reference, prep drops the superseded monopolar Raw to save RAM.
print(f'Bipolar channels: {len(result.referenced.ch_names)}')
print('First bipolar channels:', result.referenced.ch_names[:5])
print('result.raw is None after re-reference:', result.raw is None)
print(result.electrode_df[['label', 'mni_x', 'mni_y', 'mni_z', 'anode', 'cathode']].head())

## Plot one bipolar channel

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chan = result.referenced.ch_names[0]
sfreq = result.referenced.info['sfreq']
start = int(240 * sfreq)
stop = start + int(2 * sfreq)
data = result.referenced.get_data(picks=[chan])[0, start:stop]
times = np.arange(len(data)) / sfreq
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times, data, 'k', lw=0.8)
ax.set(xlabel='Time (s)', ylabel='Amplitude', title=f'Bipolar {chan}')
fig.tight_layout()
plt.show()

## CAR (trimmed) sanity check

`car_trimmed` keeps the original channel names and uses the original electrode datasheet.

In [ ]:
car_config = build_basic_pipeline_config(
    Path('../../data/sample_ieeg.fif'),
    file_format='mne',
    reference_method='car_trimmed',
    electrode_path=Path('../../data/sample_labels.xlsx'),
    preload=True,
)
car_result = run_pipeline(car_config)
print('CAR trimmed first channels:', car_result.referenced.ch_names[:5])
print('CAR trimmed contains bipolar pair names:', any('-' in ch for ch in car_result.referenced.ch_names))
print('electrode_df_referenced:', car_result.metadata.get('electrode_df_referenced'))
print(car_result.electrode_df[['label', 'mni_x', 'mni_y', 'mni_z']].head())

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# result.referenced.save(out / 'sample_bipolar_raw.fif', overwrite=True)
# result.electrode_df.to_csv(out / 'sample_labels_bp.csv', index=False)

## Next step

Chapter 04b (`04b_first_synchronization`) covers photodiode synchronization.